# Inference Notebook — Fine-tuned TinyLlama from S3
This notebook is fully standalone — just run it on its own without needing to run the training

## 1. Install Requirements

In [1]:
!pip install -q transformers peft accelerate bitsandbytes boto3


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.2 MB/s eta 0:00:00


## 2. Download Fine-tuned Model from S3

In [3]:
import boto3
import os
from google.colab import userdata

# ── Get AWS Credentials safely ───────────────────────────────────────────
AWS_ACCESS_KEY = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET_KEY = userdata.get('AWS_SECRET_ACCESS_KEY')

# Check if keys exist
if not AWS_ACCESS_KEY or not AWS_SECRET_KEY:
    raise ValueError(" AWS credentials not found in Colab Secrets")

# Remove any accidental spaces
AWS_ACCESS_KEY = AWS_ACCESS_KEY.strip()
AWS_SECRET_KEY = AWS_SECRET_KEY.strip()

# ── Config ───────────────────────────────────────────────────────────────
BUCKET    = 'cloud-project-time4'
S3_PREFIX = 'models/llama-finetuned-v4/'
LOCAL_DIR = './inference-model'

os.makedirs(LOCAL_DIR, exist_ok=True)

# ── Create S3 client correctly (no env issues) ───────────────────────────
s3 = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name='us-east-1'
)

# ── Try connection first (debug step) ────────────────────────────────────
try:
    s3.list_buckets()
    print("AWS connection successful")
except Exception as e:
    raise RuntimeError(f" AWS connection failed: {e}")

# ── Download files ──────────────────────────────────────────────────────
downloaded = []

try:
    paginator = s3.get_paginator('list_objects_v2')

    for page in paginator.paginate(Bucket=BUCKET, Prefix=S3_PREFIX):
        for obj in page.get('Contents', []):
            key = obj['Key']
            filename = os.path.basename(key)

            if not filename:
                continue

            local_path = os.path.join(LOCAL_DIR, filename)

            s3.download_file(BUCKET, key, local_path)
            downloaded.append(filename)

            print(f'Downloaded: {filename}')

    print(f'\n {len(downloaded)} file(s) ready in {LOCAL_DIR}')
    print('Files:', os.listdir(LOCAL_DIR))

except Exception as e:
    print(f" Error during download: {e}")

AWS connection successful
Downloaded: README.md
Downloaded: adapter_config.json
Downloaded: adapter_model.safetensors
Downloaded: chat_template.jinja
Downloaded: special_tokens_map.json
Downloaded: tokenizer.json
Downloaded: tokenizer.model
Downloaded: tokenizer_config.json
Downloaded: training_args.bin

 9 file(s) ready in ./inference-model
Files: ['tokenizer.model', 'tokenizer.json', 'adapter_model.safetensors', 'special_tokens_map.json', 'README.md', 'chat_template.jinja', 'training_args.bin', 'adapter_config.json', 'tokenizer_config.json']


## 3. Load Base Model + LoRA Adapter

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# Base model (4-bit — same quantization it was trained with)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4'
    ),
    device_map='auto'
)

# Attach LoRA adapter
model = PeftModel.from_pretrained(base_model, LOCAL_DIR)
model.eval()
print('Fine-tuned model loaded and ready!')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Fine-tuned model loaded and ready!


## 4. Inference Function

In [5]:
def generate_answer(problem: str, max_new_tokens: int = 256, temperature: float = 0.3) -> str:
    """
    Takes a problem and returns the answer from the fine-tuned model
    """
    # Same template it was trained with
    prompt = f'<|user|>\n{problem}</s>\n<|assistant|>\n'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only the new tokens (not the prompt)
    new_tokens = output_ids[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print('generate_answer() ready!')


generate_answer() ready!


## 5. Test — Try the Model

In [6]:
test_problems = [
    'What is 25% of 200?',
    'If a train travels at 60 km/h for 2.5 hours, how far does it go?',
    'Solve for x: 3x + 7 = 22',
]

for problem in test_problems:
    print('=' * 60)
    print(f'Problem : {problem}')
    print(f'Answer  : {generate_answer(problem)}')
print('=' * 60)


Problem : What is 25% of 200?
Answer  : To find the value of 25% of 200, we first need to find the value of 200.

200 = 200 * 100 / 100 = 20000

25% of 200 is 25 * 200 / 100 = 5000

So, the answer is:
\[ \boxed{5000} \]
Problem : If a train travels at 60 km/h for 2.5 hours, how far does it go?
Answer  : To find the distance the train goes, we need to convert the time from hours to kilometers.

The time the train travels for 2.5 hours is:
\[ 2.5 \text{ hours} = 2 \times 60 \text{ minutes} = 120 \text{ minutes} \]

The distance the train goes is:
\[ 2.5 \text{ hours} \times 60 \text{ minutes} = 120 \text{ kilometers} \]

So, the train goes \boxed{120} kilometers.
Problem : Solve for x: 3x + 7 = 22
Answer  : To solve for x, we need to find the value of x that satisfies the equation 3x + 7 = 22.

First, find the value of 22:
\[ 22 = 3x + 7 \Rightarrow 22 = 3(3x) + 7 \Rightarrow 22 = 18x + 14 \Rightarrow 22 = 34x + 14 \]

Now, substitute x into the equation 3x + 7 = 22:
\[ 3x + 7 = 22 \Righ

## 6. Try Your Own Question

In [7]:
my_problem = 'Sara bought 3 pens, and the price of each pen is 5 pounds. Then she bought a notebook for 10 pounds. How much did Sara pay in total?'  # ← change this

answer = generate_answer(my_problem)
print(f' Problem : {my_problem}')
print(f' Answer  : {answer}')


 Problem : Sara bought 3 pens, and the price of each pen is 5 pounds. Then she bought a notebook for 10 pounds. How much did Sara pay in total?
 Answer  : To find out how much Sara paid in total, we need to first calculate the total cost of the pens and the notebook.

The total cost of the pens is 3 * 5 = 15 pounds.
The total cost of the notebook is 10 * 5 = 50 pounds.

So, the total cost of the pens and the notebook is 15 + 50 = 65 pounds.

Thus, Sara paid \boxed{65} pounds in total.
